In [1]:
pip install pandas scikit-learn numpy

In [2]:
pip install pandas

In [3]:
pip install pandas numpy scikit-learn

In [4]:
pip install skl2onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.2 MB/s eta 0:00:00


In [5]:
pip install onnxmltools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.5/352.5 kB 6.8 MB/s eta 0:00:00


In [6]:
pip install imbalanced-learn

In [7]:
pip install torch torch-geometric pandas scikit-learn numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.8 MB/s eta 0:00:00


In [8]:
pip install onnx

In [38]:
import os, json, joblib, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import lightgbm as lgb

warnings.filterwarnings("ignore")

# ---------- CONFIG ----------
CSV = "/content/sample_data/aws_cloudtrail_synthetic_fixed.csv"
LABEL_COL = "trueLabel"
RANDOM_STATE = 42
N_SPLITS = 5
TARGET_ACC = 0.90
# ----------------------------

assert os.path.exists(CSV), f"CSV not found: {CSV}"
df = pd.read_csv(CSV)
print("✅ Loaded dataset:", df.shape)

# --- Cleaning & filling
df = df.fillna(0)

# --- Select features
num_features = ['bytesTransferred','hourOfDay','requestRatePerMin','riskScore']
bool_features = ['isWeekend','usualIP','usualBucket','unusualTime','highVolume','crossBucketFlow','isNormalPattern','mitigationTriggered']
cat_features  = ['eventName','userRole','geoLocation.country']

features = []
for f in num_features + bool_features + cat_features:
    if f in df.columns:
        features.append(f)

print("✅ Features used:", features)

# --- Encode target
le = LabelEncoder()
y = le.fit_transform(df[LABEL_COL].astype(str))
joblib.dump(le, "label_encoder.pkl")
print("✅ Saved label_encoder.pkl | Classes:", list(le.classes_))

# --- Prepare features
X_df = df[features].copy()
# Encode categoricals
cat_cols = [c for c in X_df.columns if X_df[c].dtype == 'object']
encoders = {}
for c in cat_cols:
    enc = LabelEncoder()
    X_df[c] = enc.fit_transform(X_df[c].astype(str))
    encoders[c] = enc
joblib.dump(encoders, "encoders.joblib")

# --- Normalize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df)
joblib.dump(scaler, "scaler.pkl")

# Save scaler stats
scaler_stats = {"mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist(), "features": features}
with open("scaler_stats.json", "w") as f:
    json.dump(scaler_stats, f, indent=2)
print("✅ Saved scaler_stats.json")

# --- SMOTE
sm = SMOTE(random_state=RANDOM_STATE)
X_bal, y_bal = sm.fit_resample(X_scaled, y)
print("✅ Label counts after SMOTE:", dict(zip(*np.unique(y_bal, return_counts=True))))

# --- Training with StratifiedKFold + early stopping callback
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
best_model, best_score = None, 0.0
fold = 0

for train_idx, val_idx in skf.split(X_bal, y_bal):
    fold += 1
    X_train, X_val = X_bal[train_idx], X_bal[val_idx]
    y_train, y_val = y_bal[train_idx], y_bal[val_idx]

    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val)

    params = {
        "objective": "multiclass",
        "num_class": len(le.classes_),
        "metric": "multi_logloss",
        "learning_rate": 0.05,
        "num_leaves": 64,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
        "min_data_in_leaf": 20,
        "lambda_l1": 1.0,
        "lambda_l2": 1.0,
        "seed": RANDOM_STATE,
        "verbosity": -1
    }

    # ✅ Modern LightGBM early stopping via callbacks
    callbacks = [
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=0)
    ]

    model = lgb.train(
        params=params,
        train_set=train_data,
        valid_sets=[train_data, val_data],
        callbacks=callbacks,
        num_boost_round=2000
    )

    preds_val = np.argmax(model.predict(X_val, num_iteration=model.best_iteration), axis=1)
    acc = accuracy_score(y_val, preds_val)
    print(f"Fold {fold}: Validation Accuracy = {acc*100:.2f}%")

    if acc > best_score:
        best_score = acc
        best_model = model

print(f"\n🏆 Best Fold Accuracy: {best_score*100:.2f}%")

# --- Final holdout test
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X_bal, y_bal, test_size=0.2, random_state=RANDOM_STATE, stratify=y_bal
)

preds = np.argmax(best_model.predict(X_test_final, num_iteration=best_model.best_iteration), axis=1)
acc_final = accuracy_score(y_test_final, preds)
print(f"\n🎯 Final Test Accuracy: {acc_final*100:.2f}%")
print("\n📜 Classification Report:")
print(classification_report(y_test_final, preds, target_names=le.classes_))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_final, preds))

# --- Save if good
if acc_final >= TARGET_ACC:
    joblib.dump(best_model, "model.pkl")
    print("\n✅ Model saved as model.pkl")

    # Optional ONNX export
    try:
        import onnxmltools
        onnx_model = onnxmltools.convert_lightgbm(best_model, name="LightGBM_Model")
        with open("model.onnx", "wb") as f:
            f.write(onnx_model.SerializeToString())
        print("✅ model.onnx exported successfully")
    except Exception as e:
        print("⚠️ ONNX export skipped:", e)
else:
    print(f"\n⚠️ Accuracy below {TARGET_ACC*100:.0f}% — achieved {acc_final*100:.2f}%")
    print("Feature importances:")
    fi = pd.Series(best_model.feature_importance(), index=features).sort_values(ascending=False)
    print(fi.head(15))


✅ Loaded dataset: (8000, 22)
✅ Features used: ['bytesTransferred', 'hourOfDay', 'requestRatePerMin', 'riskScore', 'isWeekend', 'usualIP', 'usualBucket', 'unusualTime', 'highVolume', 'crossBucketFlow', 'isNormalPattern', 'mitigationTriggered', 'eventName', 'userRole', 'geoLocation.country']
✅ Saved label_encoder.pkl | Classes: ['Credential_Theft', 'DDoS_API_Flood', 'Insider_Exfiltration', 'Normal_Traffic']
✅ Saved scaler_stats.json
✅ Label counts after SMOTE: {np.int64(0): np.int64(4000), np.int64(1): np.int64(4000), np.int64(2): np.int64(4000), np.int64(3): np.int64(4000)}
Fold 1: Validation Accuracy = 100.00%
Fold 2: Validation Accuracy = 100.00%
Fold 3: Validation Accuracy = 100.00%
Fold 4: Validation Accuracy = 100.00%
Fold 5: Validation Accuracy = 100.00%

🏆 Best Fold Accuracy: 100.00%

🎯 Final Test Accuracy: 100.00%

📜 Classification Report:
                      precision    recall  f1-score   support

    Credential_Theft       1.00      1.00      1.00       800
      DDoS_API_F

In [40]:
import joblib
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType  # ✅ correct import

# Load trained LightGBM model
model = joblib.load("/content/model.pkl")

# Define input type (number of features = 15 in your dataset)
initial_types = [("input", FloatTensorType([None, 15]))]

# Convert LightGBM to ONNX
onnx_model = onnxmltools.convert_lightgbm(model, name="LightGBM_Model", initial_types=initial_types)
with open("model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

print("✅ model.onnx exported successfully!")


✅ model.onnx exported successfully!


In [48]:
import joblib
import onnxruntime as ort
import numpy as np
import json

# ==== LOAD TRAINED ASSETS ====
model = joblib.load("/content/model.pkl")
label_encoder = joblib.load("/content/label_encoder.pkl")
with open("/content/scaler_stats.json") as f:
    scaler_stats = json.load(f)

onnx_session = ort.InferenceSession("/content/model.onnx", providers=["CPUExecutionProvider"])

# ==== FEATURE ORDER ====
features = [
    'bytesTransferred', 'hourOfDay', 'requestRatePerMin', 'riskScore', 'isWeekend',
    'usualIP', 'usualBucket', 'unusualTime', 'highVolume', 'crossBucketFlow',
    'isNormalPattern', 'mitigationTriggered', 'eventName', 'userRole', 'geoLocation.country'
]

# ==== GENERATE 10 RANDOM REALISTIC SAMPLES ====
real_time_data = np.array([
    [25000, 14, 30, 0.75, 0, 1, 1, 0, 0, 1, 1, 0, 4, 2, 1],
    [12000, 3, 12, 0.25, 1, 1, 0, 0, 0, 0, 1, 0, 7, 3, 2],
    [40000, 22, 80, 0.95, 0, 0, 1, 1, 1, 1, 0, 1, 3, 2, 5],
    [500, 11, 5, 0.10, 0, 1, 1, 0, 0, 0, 1, 0, 5, 1, 4],
    [87000, 19, 100, 0.99, 0, 0, 0, 1, 1, 1, 0, 1, 2, 3, 3],
    [900, 7, 8, 0.15, 1, 1, 1, 0, 0, 0, 1, 0, 4, 2, 1],
    [65000, 23, 60, 0.82, 0, 0, 1, 1, 1, 1, 0, 1, 8, 3, 5],
    [20000, 16, 25, 0.55, 0, 1, 0, 0, 1, 0, 1, 0, 6, 1, 2],
    [4500, 9, 10, 0.33, 1, 1, 1, 0, 0, 0, 1, 0, 1, 2, 3],
    [30000, 20, 45, 0.70, 0, 0, 1, 1, 1, 1, 0, 1, 3, 3, 5]
])

# ==== APPLY SCALING ====
X_scaled = (real_time_data - np.array(scaler_stats["mean"])) / np.array(scaler_stats["scale"])
X_scaled = X_scaled.astype(np.float32)  # ✅ Convert to float32 for ONNX

# ==== PKL PREDICTIONS ====
pred_pkl_proba = model.predict(X_scaled)
pred_pkl_labels = np.argmax(pred_pkl_proba, axis=1)

# ==== ONNX PREDICTIONS ====
input_name = onnx_session.get_inputs()[0].name
onnx_output = onnx_session.run(None, {input_name: X_scaled})[0]

# Handle both probability and direct label outputs
if onnx_output.ndim == 2:
    pred_onnx_proba = onnx_output
    pred_onnx_labels = np.argmax(pred_onnx_proba, axis=1)
else:
    pred_onnx_labels = onnx_output.astype(int)
    pred_onnx_proba = None

# ==== MAP CLASS NAMES ====
label_names = label_encoder.classes_
pred_pkl_names = [label_names[p] for p in pred_pkl_labels]
pred_onnx_names = [label_names[p] for p in pred_onnx_labels]

# ==== COMPARE ====
print("\n🧩 Model Prediction Comparison (PKL vs ONNX)\n")
for i, (p1, p2, n1, n2) in enumerate(zip(pred_pkl_labels, pred_onnx_labels, pred_pkl_names, pred_onnx_names)):
    match = "✅" if p1 == p2 else "❌"
    print(f"Sample {i+1:>2} | PKL: {n1:<20} | ONNX: {n2:<20} | {match}")

if pred_onnx_proba is not None:
    diff = np.abs(pred_pkl_proba - pred_onnx_proba).mean()
    print(f"\nAverage Probability Difference: {diff:.8f}")
else:
    print("\nℹ️ ONNX model returned direct class predictions (no probability output).")



🧩 Model Prediction Comparison (PKL vs ONNX)

Sample  1 | PKL: Credential_Theft     | ONNX: Credential_Theft     | ✅
Sample  2 | PKL: Normal_Traffic       | ONNX: Normal_Traffic       | ✅
Sample  3 | PKL: Insider_Exfiltration | ONNX: Insider_Exfiltration | ✅
Sample  4 | PKL: Normal_Traffic       | ONNX: Normal_Traffic       | ✅
Sample  5 | PKL: Insider_Exfiltration | ONNX: Insider_Exfiltration | ✅
Sample  6 | PKL: Normal_Traffic       | ONNX: Normal_Traffic       | ✅
Sample  7 | PKL: Insider_Exfiltration | ONNX: Insider_Exfiltration | ✅
Sample  8 | PKL: Credential_Theft     | ONNX: Credential_Theft     | ✅
Sample  9 | PKL: Normal_Traffic       | ONNX: Normal_Traffic       | ✅
Sample 10 | PKL: Insider_Exfiltration | ONNX: Insider_Exfiltration | ✅

ℹ️ ONNX model returned direct class predictions (no probability output).
